# Stylepal RAG Evaluation with Ragas

Evaluate the Stylepal RAG pipeline (retrieval + generation) using Ragas metrics:
- **Faithfulness**: Is the answer grounded in the retrieved context?
- **Answer Relevancy**: Is the answer relevant to the question?
- **Context Precision**: Are the retrieved contexts relevant?
- **Context Recall**: Does retrieval capture the necessary information?

Based on AIE9/10_Evaluating_RAG_With_Ragas patterns.

## Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "eval":
    project_root = project_root.parent
backend_dir = project_root / "backend"
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from dotenv import load_dotenv
load_dotenv(project_root / ".env", override=True)
_ = load_dotenv(backend_dir / ".env", override=True)

## Shared: RAG Pipeline & Evaluator

Run this section once. Both Original and Extended evaluation paths use these.

In [23]:
import os
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, ContextPrecision, NoiseSensitivity

from services import rag

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0, google_api_key=os.getenv("GEMINI_API_KEY"))
QA_PROMPT = ChatPromptTemplate.from_template(
    """Answer the question based only on the context below. Be concise.

Context:
{context}

Question: {question}

Answer:"""
)

def run_rag_pipeline(question: str, model: str = "gemini-2.5-flash") -> tuple[str, list[str]]:
    """Retrieve + generate. model: gemini-2.5-flash (default), gemini-2.5-pro, or openai (gpt-4.1-mini)."""
    if model == "openai":
        from pathlib import Path
        from langchain_openai import ChatOpenAI
        _root = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
        env_file = _root / ".env"
        api_key = None
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                if line.strip().startswith("OPENAI_API_KEY="):
                    api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
                    break
        if not api_key:
            raise ValueError("OPENAI_API_KEY not found in project root .env")
        _llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=api_key)
    elif model == "gemini-2.5-flash":
        _llm = llm
    else:
        _llm = ChatGoogleGenerativeAI(model=model, temperature=0, google_api_key=os.getenv("GEMINI_API_KEY"))
    docs = rag.retrieve(question, top_k=5)
    contexts = [d["content"] for d in docs]
    context_str = "\n\n".join(contexts) if contexts else "(No context retrieved)"
    chain = QA_PROMPT | _llm | StrOutputParser()
    answer = chain.invoke({"context": context_str, "question": question})
    return answer, contexts

# Evaluator: Gemini (Original set uses this; results already saved)
evaluator_llm = LangchainLLMWrapper(llm)
evaluator_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=os.getenv("GEMINI_API_KEY"))
)

/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/1271650905.py:9: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, ContextPrecision, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/1271650905.py:9: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, ContextPrecision, NoiseSensitivity
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/1271650905.py:9: Deprec

## Original Evaluation

Curated dataset: questions from style knowledge (Flattering Fashions) with reference answers.

In [14]:
EVAL_SAMPLES = [
    {
        "user_input": "What do horizontal lines do to the body?",
        "reference_contexts": ["Horizontal lines move the eye across the body, so they appear to add width and make a person appear heavier and shorter."],
        "reference": "Horizontal lines draw the eye across the body and appear to add width, making a person look heavier and shorter.",
    },
    {
        "user_input": "What is the hourglass silhouette?",
        "reference_contexts": ["Hourglass: Feminine silhouette emphasizes bust and hips and highlights a small waist."],
        "reference": "The hourglass silhouette emphasizes bust and hips with a small waist, creating a feminine shape.",
    },
    {
        "user_input": "How can color be used to minimize size?",
        "reference_contexts": ["When light and dark colors are worn together, dark clothes minimize, while light clothes emphasize."],
        "reference": "Dark colors minimize size when worn with light colors; light colors emphasize.",
    },
    {
        "user_input": "What do vertical lines do for appearance?",
        "reference_contexts": ["Vertical lines lead the eye up and down so they add height to the body and make it appear narrower."],
        "reference": "Vertical lines lead the eye up and down, adding height and making the body appear narrower.",
    },
    {
        "user_input": "What is the golden mean in fashion?",
        "reference_contexts": ["Golden Mean Belief that 3:5 proportions and 5:8 proportions are pleasing to the eye."],
        "reference": "The golden mean is the belief that 3:5 and 5:8 proportions are pleasing to the eye.",
    },
]

## Original: Run RAG Pipeline

In [27]:
results = []
for sample in EVAL_SAMPLES:
    answer, contexts = run_rag_pipeline(sample["user_input"])
    results.append({
        "user_input": sample["user_input"],
        "reference_contexts": sample["reference_contexts"],
        "reference": sample["reference"],
        "response": answer,
        "retrieved_contexts": contexts,
    })

df = pd.DataFrame(results)
df

,user_input,reference_contexts,reference,response,retrieved_contexts
0,What do horizontal lines do to the body?,[Horizontal lines move the eye across the body...,Horizontal lines draw the eye across the body ...,Horizontal lines add width and make a person a...,[Line \n \n• Horizontal lines move the eye acr...
1,What is the hourglass silhouette?,[Hourglass: Feminine silhouette emphasizes bus...,The hourglass silhouette emphasizes bust and h...,The hourglass silhouette emphasizes bust and h...,[Hourglass\nYou are an Hourglass if..\n..your ...
2,How can color be used to minimize size?,"[When light and dark colors are worn together,...",Dark colors minimize size when worn with light...,"Dark clothes minimize, and monochromatic dress...",[• How to create pleasing proportions and avoi...
3,What do vertical lines do for appearance?,[Vertical lines lead the eye up and down so th...,"Vertical lines lead the eye up and down, addin...","Vertical lines lead the eye up and down, addin...",[Line \n \n• Horizontal lines move the eye acr...
4,What is the golden mean in fashion?,[Golden Mean Belief that 3:5 proportions and 5...,The golden mean is the belief that 3:5 and 5:8...,The Golden Mean is the belief that 3:5 and 5:8...,[• Crew neck High plain neckline with knit ri...


## Original: Evaluate with Ragas

In [28]:
# evaluator_llm, evaluator_embeddings from Shared section (cell 5)
eval_dataset = EvaluationDataset.from_pandas(df)
result = evaluate(
    dataset=eval_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), ContextPrecision(), NoiseSensitivity()],
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    run_config=RunConfig(timeout=360),
)
result.scores if hasattr(result, "scores") else result

Evaluating:   0%|          | 0/35 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


[{'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.67),
  'answer_relevancy': np.float64(0.8700238359102466),
  'context_entity_recall': 0.0,
  'context_precision': 0.9999999999,
  'noise_sensitivity(mode=relevant)': 0.0},
 {'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.8),
  'answer_relevancy': np.float64(0.8297194708989012),
  'context_entity_recall': 0.0,
  'context_precision': 0.8874999999778125,
  'noise_sensitivity(mode=relevant)': 0.0},
 {'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.0),
  'answer_relevancy': np.float64(0.6633978517063146),
  'context_entity_recall': 0.0,
  'context_precision': 0.49999999995,
  'noise_sensitivity(mode=relevant)': 0.25},
 {'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(1.0),
  'answer_relevancy': np.float64(0.8458425778431088),
  'context_entity_recall': 0.0,
  'conte

## RAG Evaluation Summary – Original Set (5 samples)

### What worked well

- **Retrieval (context recall): 100%** – All 5 samples retrieved the needed context.
- **Faithfulness: 100%** – All answers stayed grounded in the retrieved context.
- **Context precision: ~0.82 avg** – Relevant chunks are generally ranked high; samples 1, 2, and 5 reach 0.89–1.0.
- **Answer relevancy: 0.66–0.87** – Most answers were relevant; 4 of 5 were above 0.82.
- **Factual correctness** – 2 samples (4 and 5) reached 1.0; 2 others (1 and 2) were 0.67 and 0.8.

### What didn't work well

- **Sample 3** – Clear outlier:
  - Factual correctness: **0.0**
  - Answer relevancy: **0.66** (lowest)
  - Context precision: **0.50** – Irrelevant or noisy chunks ranked high; retrieval surfaces wrong content for "color minimize size."
  - Noise sensitivity: **0.25** (only sample with non-zero noise sensitivity)
  
  This corresponds to "How can color be used to minimize size?" – retrieval returns proportion/avoidance content instead of the color-minimize rule; the model adds monochromatic dressing not in the reference.

- **Context entity recall: 0 for all samples** – Either the metric doesn't fit your reference format, or entities in the reference are not being matched in the retrieved context.

### Takeaways

| Metric              | Avg / pattern | Interpretation                          |
|---------------------|---------------|-----------------------------------------|
| Context recall      | 1.0           | Retrieval is strong                     |
| Context precision   | ~0.82         | Generally good; sample 3 weak (0.50)    |
| Faithfulness        | 1.0           | No hallucination beyond retrieved text  |
| Factual correctness | ~0.69         | Mixed; one sample at 0.0                |
| Answer relevancy    | ~0.80         | Generally good; one weaker sample       |
| Context entity recall | 0.0         | Likely metric or format mismatch         |

**Recommendation:** Inspect sample 3 (color/minimize size) in detail: compare the model answer to the reference and to the retrieved chunks. Improve retrieval or add a dedicated chunk for the color-minimize rule; consider broadening the reference or adjusting the prompt.

---
## Extended Evaluation

Separate path: additional style knowledge questions. Run this section independently.

In [3]:
EVAL_SAMPLES_EXTENDED = [
    {
        "user_input": "What do horizontal lines do to the body?",
        "reference_contexts": ["Horizontal lines move the eye across the body, so they appear to add width and make a person appear heavier and shorter."],
        "reference": "Horizontal lines draw the eye across the body and appear to add width, making a person look heavier and shorter.",
    },
    {
        "user_input": "What is the hourglass silhouette?",
        "reference_contexts": ["Hourglass: Feminine silhouette emphasizes bust and hips and highlights a small waist."],
        "reference": "The hourglass silhouette emphasizes bust and hips with a small waist, creating a feminine shape.",
    },
    {
        "user_input": "What necklines work for an hourglass body type?",
        "reference_contexts": ["Hourglass: Feminine silhouette emphasizes bust and hips. V-neck, scoop neck, and sweetheart necklines flatter by balancing the bust and highlighting the waist."],
        "reference": "V-neck, scoop neck, and sweetheart necklines work well for hourglass figures by balancing the bust and highlighting the waist.",
    },
    {
        "user_input": "What colors pair well with navy?",
        "reference_contexts": ["Navy pairs well with white, cream, gray, and burgundy. Avoid pairing navy with black in the same outfit for a cohesive look."],
        "reference": "Navy pairs well with white, cream, gray, and burgundy.",
    },
    {
        "user_input": "How should I dress for a pear-shaped body?",
        "reference_contexts": ["Pear shape: hips wider than shoulders. Emphasize the upper body with details, structured shoulders, and darker bottoms to minimize hips."],
        "reference": "Emphasize the upper body with details and structured shoulders; use darker bottoms to balance wider hips.",
    },
    {
        "user_input": "What waistlines flatter an apple shape?",
        "reference_contexts": ["Apple shape: fuller midsection. Empire waist, A-line, and wrap styles create definition. Avoid tight waistbands."],
        "reference": "Empire waist, A-line, and wrap styles flatter apple shapes by creating definition; avoid tight waistbands.",
    },
]

In [14]:
# Extended only: use OpenAI for evaluator (avoids Gemini TPM exhaustion)
from pathlib import Path
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

_root = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
env_file = _root / ".env"
api_key = None
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.strip().startswith("OPENAI_API_KEY="):
            api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
            break
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in project root .env")
print(f"key starts with: {api_key[:4] if api_key else 'None'}...")
evaluator_llm_ext = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=api_key))
evaluator_embeddings_ext = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key))

key starts with: sk-p...


/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/1623238964.py:16: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm_ext = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=api_key))
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/1623238964.py:17: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings_ext = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key))


In [5]:
# Run RAG pipeline on extended dataset (Gemini 2.5 Pro)
results_ext = []
for sample in EVAL_SAMPLES_EXTENDED:
    answer, contexts = run_rag_pipeline(sample["user_input"], model="gemini-2.5-pro")
    results_ext.append({
        "user_input": sample["user_input"],
        "reference_contexts": sample["reference_contexts"],
        "reference": sample["reference"],
        "response": answer,
        "retrieved_contexts": contexts,
    })

df_ext = pd.DataFrame(results_ext)
df_ext

,user_input,reference_contexts,reference,response,retrieved_contexts
0,What do horizontal lines do to the body?,[Horizontal lines move the eye across the body...,Horizontal lines draw the eye across the body ...,"Horizontal lines move the eye across the body,...",[Line \n \n• Horizontal lines move the eye acr...
1,What is the hourglass silhouette?,[Hourglass: Feminine silhouette emphasizes bus...,The hourglass silhouette emphasizes bust and h...,A feminine silhouette that emphasizes the bust...,[Hourglass\nYou are an Hourglass if..\n..your ...
2,What necklines work for an hourglass body type?,[Hourglass: Feminine silhouette emphasizes bus...,"V-neck, scoop neck, and sweetheart necklines w...",V and scoop necklines.,[Hourglass\nYou are an Hourglass if..\n..your ...
3,What colors pair well with navy?,"[Navy pairs well with white, cream, gray, and ...","Navy pairs well with white, cream, gray, and b...","Based on the context provided, there is no inf...",[Color Theory in Fashion\nColors play a crucia...
4,How should I dress for a pear-shaped body?,[Pear shape: hips wider than shoulders. Emphas...,Emphasize the upper body with details and stru...,"Try structured tops, A-line skirts, and straig...","[common types:\nPear-shaped: Narrow shoulders,..."
5,What waistlines flatter an apple shape?,"[Apple shape: fuller midsection. Empire waist,...","Empire waist, A-line, and wrap styles flatter ...",High-waist.,"[common types:\nPear-shaped: Narrow shoulders,..."


In [9]:
# Debug: run agent for first query and inspect tool_calls
# Step 1: Quick direct LLM test (~10-30 sec) - does Gemini return tool_calls?
print("--- Direct LLM test (Gemini + bind_tools + tool_choice=any) ---")
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from services.agent import STYLIST_TOOLS
_llm = ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0, google_api_key=__import__('os').getenv("GEMINI_API_KEY"))
_llm_tools = _llm.bind_tools(STYLIST_TOOLS, tool_choice="any")
_direct = _llm_tools.invoke([HumanMessage(content="What should I wear for a job interview tomorrow?")])
print("Direct response tool_calls:", getattr(_direct, 'tool_calls', None))
print("Direct response has tool_calls:", bool(getattr(_direct, 'tool_calls', None)))



--- Direct LLM test (Gemini + bind_tools + tool_choice=any) ---
Direct response tool_calls: [{'name': 'get_wardrobe', 'args': {}, 'id': '3cbebeeb-5502-4f41-875d-12a72106a815', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'question': 'what is the weather tomorrow?'}, 'id': 'e51c4de4-e9d1-4943-b354-a2199df679ee', 'type': 'tool_call'}]
Direct response has tool_calls: True


## Extended: Evaluate with Ragas

In [15]:
# Evaluate extended RAG dataset (OpenAI evaluator - avoids Gemini TPM exhaustion)
eval_dataset_ext = EvaluationDataset.from_pandas(df_ext)
result_ext = evaluate(
    dataset=eval_dataset_ext,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), ContextPrecision(), NoiseSensitivity()],
    llm=evaluator_llm_ext,
    embeddings=evaluator_embeddings_ext,
    run_config=RunConfig(timeout=600, max_workers=4),
)
result_ext.scores if hasattr(result_ext, "scores") else result_ext

Evaluating:   0%|          | 0/42 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


[{'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(1.0),
  'answer_relevancy': np.float64(0.8670422740219837),
  'context_entity_recall': 0.9999999900000002,
  'context_precision': 0.9999999999,
  'noise_sensitivity(mode=relevant)': 0.0},
 {'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(1.0),
  'answer_relevancy': np.float64(0.5250422249373354),
  'context_entity_recall': 0.0,
  'context_precision': 0.8874999999778125,
  'noise_sensitivity(mode=relevant)': 0.0},
 {'context_recall': 0.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.44),
  'answer_relevancy': np.float64(0.6071329731661061),
  'context_entity_recall': 0.0,
  'context_precision': 0.9999999999,
  'noise_sensitivity(mode=relevant)': 0.0},
 {'context_recall': 0.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.0),
  'answer_relevancy': np.float64(0.0),
  'context_entity_recall': 0.0,
  'context


## RAG Evaluation Summary – Extended Set (6 samples)

### What worked well

- **Samples 1 & 2** (horizontal lines, hourglass silhouette): Context recall 1.0, faithfulness 1.0, factual correctness 1.0, context precision 0.89–1.0. Strong performance.
- **Sample 5** (pear-shaped body): Context recall 1.0, context precision 1.0 – retrieval finds relevant content; faithfulness 0.8, factual 0.25.
- **Faithfulness: 5/6 at 1.0** – Most answers stay grounded in retrieved context.
- **Context precision** – 4 of 6 samples reach 0.89–1.0 when retrieval succeeds; sample 3 (necklines) also 1.0 despite context recall 0.
- **Context entity recall** – Sample 1 reaches ~1.0; others at 0.

### What didn't work well

- **Context recall: 0.0 for samples 3, 4, 6** – Retrieval fails for:
  - Necklines for hourglass (sample 3)
  - Colors that pair with navy (sample 4)
  - Waistlines for apple shape (sample 6)

- **Sample 4** (navy colors) – Worst: context recall 0, factual 0, answer relevancy 0, context precision 0. No useful retrieval or answer.

- **Sample 5** (pear-shaped) – Faithfulness 0.8, factual 0.25, answer relevancy 0.42, noise sensitivity 0.4. Retrieval works but answer quality is mixed.

- **Sample 6** (apple waistlines) – Context recall 0, factual 0, answer relevancy 0.35, context precision 0.70, noise sensitivity 1.0. Retrieval failed.

### Comparison: Original vs Extended

| Metric                | Original (5) | Extended (6) |
|-----------------------|--------------|--------------|
| Context recall        | 100% (5/5)   | 50% (3/6)    |
| Context precision     | ~0.82 avg    | ~0.76 avg    |
| Faithfulness          | 100%         | 83% (5 at 1.0, 1 at 0.8) |
| Factual correctness   | ~0.69 avg    | ~0.45 avg    |
| Answer relevancy      | ~0.80 avg    | ~0.46 avg    |
| Context entity recall | 0.0          | ~0.17 (1 sample ~1.0) |
| Noise sensitivity     | ~0.05 avg    | ~0.23 avg    |

### Takeaways

1. **Retrieval improved for pear-shaped** – Sample 5 now has context recall 1.0.
2. **KB coverage gaps remain** – Navy color pairing and apple-shape waistlines still uncovered; necklines for hourglass not retrieved.
3. **Context precision** – When retrieval succeeds, precision is high (0.89–1.0); navy (0) and apple (0.70) drag the average.
4. **Sample 5 trade-off** – When retrieval succeeds, answer quality (factual, relevancy) can be lower; noise sensitivity 0.4 suggests some ungrounded content.
5. **Faithfulness** – One sample (5) at 0.8; others grounded.

**Recommendation:** Add content for navy color pairing and apple-shape styling. Review sample 5's answer for drift or hallucination when retrieval succeeds.


---
## Synthetic Data Generation (Abstracted Approach)

Generate synthetic eval samples from style knowledge documents using Ragas' abstracted SDG workflow (AIE9/09_Synthetic_Data_Generation). Builds knowledge graph under the hood, applies default transformations, and generates diverse questions.

### Load style knowledge documents

Same source as RAG: PDFs from `data/style_knowledge/` (or `backend/data/style_knowledge/`). Chunk with same settings as seed_rag.

In [16]:
# Run once if needed: !pip install nltk
import nltk
nltk.download("punkt", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Style knowledge dir (same as seed_rag)
style_dir = project_root / "data" / "style_knowledge"
if not style_dir.exists():
    style_dir = project_root / "backend" / "data" / "style_knowledge"

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs_sdg = []
for pdf_path in sorted(style_dir.glob("*.pdf")):
    loader = PyPDFLoader(str(pdf_path))
    raw = loader.load()
    chunks = text_splitter.split_documents(raw)
    for doc in chunks:
        if doc.metadata.get("source"):
            doc.metadata["source"] = str(Path(doc.metadata["source"]).name)
    docs_sdg.extend(chunks)

print(f"Loaded {len(docs_sdg)} chunks from {len(list(style_dir.glob('*.pdf')))} PDFs")

Loaded 132 chunks from 4 PDFs


In [18]:
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Use Gemini for SDG (same provider as RAG)
generator_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(model="gemini-2.5-pro", temperature=0, google_api_key=os.getenv("GEMINI_API_KEY"))
)
generator_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=os.getenv("GEMINI_API_KEY"))
)

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset_sdg = generator.generate_with_langchain_docs(docs_sdg, testset_size=10)

/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/1913640658.py:7: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/1913640658.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(


Applying SummaryExtractor:   0%|          | 0/71 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/132 [00:00<?, ?it/s]

Node 8b114b5a-9567-4d43-a34e-e82992a5453b does not have a summary. Skipping filtering.
Node 3c65a66a-a9e3-444d-a72c-4e96ffd67f3c does not have a summary. Skipping filtering.
Node 172e52e5-09fb-4de0-b91c-a88d0132acd0 does not have a summary. Skipping filtering.
Node efe993d5-06f2-4ca3-800b-d898591408dd does not have a summary. Skipping filtering.
Node 8dd0c19f-f994-435c-aba5-2d28b0140d1c does not have a summary. Skipping filtering.
Node a912a95a-9cb5-4f8e-8d4f-9e7028a7abcb does not have a summary. Skipping filtering.
Node ad4344e5-f89e-40fb-b68c-a3343a28cdf2 does not have a summary. Skipping filtering.
Node c8d6393f-d683-46dc-abd7-29d12dfc828d does not have a summary. Skipping filtering.
Node 7f81537c-b5eb-4205-bff0-5b7fcf1c0081 does not have a summary. Skipping filtering.
Node f90ca5b3-fa0f-4ca7-9aa0-5ac7fdd606b9 does not have a summary. Skipping filtering.
Node fe1a9caa-66da-44ab-b0e3-8f1201b0afc6 does not have a summary. Skipping filtering.
Node 38e2589e-c7e8-4a14-b4bd-52436720ac17 d

Applying EmbeddingExtractor:   0%|          | 0/71 [00:00<?, ?it/s]

...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [19]:
# Convert synthetic dataset to eval format and preview
df_sdg = dataset_sdg.to_pandas()
df_sdg

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,"According to the guide that begins with ""Bonjo...",[How To Use This Guide\nBonjour Femme!\nThe tr...,The trick to looking your best in everything t...,Maya Singh,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
1,i always thought i was a pear shape cuz of my ...,[How To Use This Guide\nOften times when worki...,The guide says that often times when working w...,Maya Singh,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
2,i think im an hourglass shape cause my hips an...,[Hourglass\nYou are an Hourglass if..\n..your ...,Your main style goal is to keep the natural ba...,Maya Singh,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
3,"Based on these style guidelines, what is the r...",[to areas you don’t love. Button \ndowns with...,Button-down shirts with princess seams are sai...,Maya Singh,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
4,"Based on the principle of visual design, how d...",[<1-hop>\n\njeans are typically best as they \...,"According to the principle of visual design, w...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,"As part of the Y2K revivals trend, what is the...",[<1-hop>\n\njeans are typically best as they \...,"As part of the Y2K revivals trend, the most fl...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,"So like, if understanding my Body Shapes is ke...",[<1-hop>\n\njeans are typically best as they \...,Understanding your body shape is important for...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,how can balancing body proportions with clothe...,[<1-hop>\n\njeans are typically best as they \...,"According to the provided text, using clothing...",NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,ok so i think my body is an inverted triangle ...,[<1-hop>\n\nInverted Triangle\nYou are an Inve...,"For an Inverted Triangle body shape, where you...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,I'm trying to find more Flattering Fashions on...,[<1-hop>\n\nIgnore them and risk looking out o...,"The ""Flattering Fashions"" program teaches how ...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer


### Run RAG pipeline on synthetic data and evaluate

In [24]:
# Run RAG pipeline on synthetic dataset (OpenAI – avoids Gemini TPM exhaustion)
results_sdg = []
for _, row in df_sdg.iterrows():
    answer, contexts = run_rag_pipeline(row["user_input"], model="openai")
    results_sdg.append({
        "user_input": row["user_input"],
        "reference_contexts": row["reference_contexts"],
        "reference": row["reference"],
        "response": answer,
        "retrieved_contexts": contexts,
    })

df_sdg_eval = pd.DataFrame(results_sdg)
df_sdg_eval

,user_input,reference_contexts,reference,response,retrieved_contexts
0,"According to the guide that begins with ""Bonjo...",[How To Use This Guide\nBonjour Femme!\nThe tr...,The trick to looking your best in everything t...,The essential trick to looking your best in al...,[How To Use This Guide\nBonjour Femme!\nThe tr...
1,i always thought i was a pear shape cuz of my ...,[How To Use This Guide\nOften times when worki...,The guide says that often times when working w...,The guide says that clients who think they hav...,[How To Use This Guide\nOften times when worki...
2,i think im an hourglass shape cause my hips an...,[Hourglass\nYou are an Hourglass if..\n..your ...,Your main style goal is to keep the natural ba...,The main style goal for your hips is to keep t...,[Hourglass\nYou are an Hourglass if..\n..your ...
3,"Based on these style guidelines, what is the r...",[to areas you don’t love. Button \ndowns with...,Button-down shirts with princess seams are sai...,Wear the button-down shirt with the top 2 butt...,[10 Universal Style Rules\n1. V or scoop neckl...
4,"Based on the principle of visual design, how d...",[<1-hop>\n\njeans are typically best as they \...,"According to the principle of visual design, w...",A-line skirts help flatter pear-shaped bodies ...,[and is a good choice for anyone carrying weig...
5,"As part of the Y2K revivals trend, what is the...",[<1-hop>\n\njeans are typically best as they \...,"As part of the Y2K revivals trend, the most fl...",The most flattering way to wear boot cut jeans...,[work for you). But the right boot cut jean \n...
6,"So like, if understanding my Body Shapes is ke...",[<1-hop>\n\njeans are typically best as they \...,Understanding your body shape is important for...,Glance helps by using AI to study your body ty...,[What Makes It Game-Changing?\nAI-Tailored Rec...
7,how can balancing body proportions with clothe...,[<1-hop>\n\njeans are typically best as they \...,"According to the provided text, using clothing...",Balancing body proportions with clothes is abo...,"[common types:\nPear-shaped: Narrow shoulders,..."
8,ok so i think my body is an inverted triangle ...,[<1-hop>\n\nInverted Triangle\nYou are an Inve...,"For an Inverted Triangle body shape, where you...",Boot cut pants are good for an inverted triang...,[Inverted Triangle\nYou are an Inverted triang...
9,I'm trying to find more Flattering Fashions on...,[<1-hop>\n\nIgnore them and risk looking out o...,"The ""Flattering Fashions"" program teaches how ...",Asymmetrical Balance helps create visual inter...,[naturally drawn to balance. Nature is natural...


In [25]:
# SDG: use OpenAI for evaluator (avoids Gemini TPM exhaustion; same as Extended)
from pathlib import Path
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

_root = Path.cwd().parent if Path.cwd().name == "eval" else Path.cwd()
env_file = _root / ".env"
api_key = None
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.strip().startswith("OPENAI_API_KEY="):
            api_key = line.split("=", 1)[1].strip().strip('"').strip("'")
            break
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in project root .env")

evaluator_llm_sdg = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=api_key))
evaluator_embeddings_sdg = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key))

/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/3150407504.py:18: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm_sdg = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini", temperature=0, api_key=api_key))
/var/folders/th/xr8r9j1909g42k60t28wfqjw0000gn/T/ipykernel_12702/3150407504.py:19: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings_sdg = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key))


In [27]:
# Evaluate synthetic RAG dataset (OpenAI evaluator – avoids Gemini TPM exhaustion)
eval_dataset_sdg = EvaluationDataset.from_pandas(df_sdg_eval)
result_sdg = evaluate(
    dataset=eval_dataset_sdg,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), ContextPrecision(), NoiseSensitivity()],
    llm=evaluator_llm_sdg,
    embeddings=evaluator_embeddings_sdg,
    run_config=RunConfig(timeout=360, max_workers=4),
)
result_sdg.scores if hasattr(result_sdg, "scores") else result_sdg

Evaluating:   0%|          | 0/84 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


[{'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.57),
  'answer_relevancy': np.float64(0.8115445100547832),
  'context_entity_recall': 0.0,
  'context_precision': 0.8666666666377778,
  'noise_sensitivity(mode=relevant)': 0.0},
 {'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.75),
  'answer_relevancy': np.float64(0.8829866686690154),
  'context_entity_recall': 0.0,
  'context_precision': 0.9999999999,
  'noise_sensitivity(mode=relevant)': 0.0},
 {'context_recall': 1.0,
  'faithfulness': 0.8571428571428571,
  'factual_correctness(mode=f1)': np.float64(0.6),
  'answer_relevancy': np.float64(0.733076449602723),
  'context_entity_recall': 0.0,
  'context_precision': 0.9999999999,
  'noise_sensitivity(mode=relevant)': 0.14285714285714285},
 {'context_recall': 1.0,
  'faithfulness': 1.0,
  'factual_correctness(mode=f1)': np.float64(0.67),
  'answer_relevancy': np.float64(0.7177192305347769),
  'context


## RAG Evaluation Summary – Synthetic Set (12 samples)

### Metrics overview

| Metric | Min | Max | Avg (approx) |
|--------|-----|-----|---------------|
| Context recall | 0.0 | 1.0 | ~0.85 |
| Context precision | 0.33 | 1.0 | ~0.77 |
| Faithfulness | 0.33 | 1.0 | ~0.79 |
| Factual correctness | 0.0 | 0.75 | ~0.55 |
| Answer relevancy | 0.60 | 0.96 | ~0.78 |
| Context entity recall | 0.0 | 1.0 | ~0.25 |
| Noise sensitivity | 0.0 | 1.0 | ~0.24 |

### Strong samples (1, 2, 4, 12)

- Context recall: 1.0  
- Faithfulness: 1.0  
- Factual correctness: 0.57–0.75  
- Answer relevancy: 0.72–0.88  

### Weak samples

- **Sample 5**: Context recall 0.5, faithfulness 0.33, factual 0.0 – worst case; retrieval and answer quality both poor.  
- **Sample 6**: Faithfulness 0.4 despite recall 1.0 – answer drifts from context.  
- **Sample 7**: Context recall 0.67, noise sensitivity 0.7 – retrieval gap and noisy chunks.  
- **Sample 8**: Context recall 0.0 – retrieval failed; faithfulness 0.75, factual 0.67 from partial context.  
- **Sample 9**: Faithfulness 0.86, factual 0.5, noise 0.29 – mixed quality.  
- **Sample 10**: Faithfulness 0.5, factual 0.5 – low grounding.  
- **Sample 12**: Faithfulness 1.0 but noise sensitivity 1.0 – answer sensitive to irrelevant content.  

### Takeaways

1. **Retrieval gaps** – Samples 5, 7, 8 have context recall 0–0.67; sample 8 has zero retrieval.
2. **Faithfulness** – Samples 5, 6, 10 drop to 0.33–0.5; suggests hallucination or drift when retrieval is weak.
3. **Context precision** – Samples 4, 5, 6 at 0.33–0.5 indicate noisy or irrelevant chunks ranked high.
4. **Synthetic diversity** – SDG surfaces harder cases (multi-hop, edge topics) that the curated sets did not.
5. **Context entity recall** – Non-zero in 6 of 12 samples; samples 4, 10, 12 reach 0.5–1.0.

### Comparison: Original vs Extended vs Synthetic (12)

| Metric | Original (5) | Extended (6) | Synthetic (12) |
|--------|--------------|--------------|----------------|
| Context recall | 100% | 50% | 75% (9/12 at 1.0) |
| Context precision | ~0.82 | ~0.76 | ~0.77 |
| Faithfulness | 100% | 83% | ~79% |
| Factual correctness | ~0.69 | ~0.45 | ~0.55 |
| Answer relevancy | ~0.80 | ~0.46 | ~0.78 |
| Context entity recall | 0.0 | ~0.17 | ~0.25 |
| Noise sensitivity | ~0.05 | ~0.23 | ~0.24 |

Synthetic questions are harder than Original but better covered than Extended. Samples 5, 7, 8 are good candidates for manual review and KB/retrieval improvements.